## 45 Days ML Journey - Day02

### Data cleaning

In [1]:
# import necessary libs.

import numpy as np
import pandas as pd

### Loading the same titanic dataset.

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/rashida048/Datasets/master/titanic_data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


### First check for duplicate rows.
If any duplicate row present then drop them.

In [3]:
df.duplicated().sum()

np.int64(0)

In our case there is no duplicate row present, If there are any then we use
df.drop_duplicates() function

### Second, we'll check for missing or `nan` values.

In [4]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

We can clearly see that columns like `Age`, `Cabin` and `Embarked` contains missing values.

### Strategy for filling missing values

For **Embarked** where only 2 values are missing, we fill these value with the most frequent value of corresponding column, As it follows the MCAR (Missing Completely At Random) concept.

In [5]:
print("Results of Embarked column (BEFORE) : ", df.Embarked.isna().sum())

most_frequent_value = df.Embarked.value_counts(ascending=False).reset_index().iloc[0].Embarked
df["Embarked"] = df["Embarked"].fillna(value=most_frequent_value)

print("Results of Embarked column (AFTER) : ", df.Embarked.isna().sum())

Results of Embarked column (BEFORE) :  2
Results of Embarked column (AFTER) :  0


Below given code segment shows that approx 77% of values in **Cabin** are missing. It's better if we drop this column, because it violated MCAR.

In [6]:
print("Percentage of value missing in `Cabin` column : ", (df.Cabin.isna().sum() / df.shape[0])*100)

Percentage of value missing in `Cabin` column :  77.10437710437711


In [7]:
df.drop(columns=["Cabin"], inplace=True)

Last we've **Age** column, we can fill this with variety of values. Here, for demonstration purpose we're using `mean` of corresponding column.

In [8]:
print("Results of Age column (BEFORE) : ", df.Embarked.isna().sum())

df["Age"] = df["Age"].fillna(df["Age"].mean())

print("Results of Age column (AFTER) : ", df.Embarked.isna().sum())

Results of Age column (BEFORE) :  0
Results of Age column (AFTER) :  0


### Third, Selecting only relevant columns
Here, columns like `PassengerId`, `Ticket`, `Name` are irrelevant columns.
It's better to drop them

In [9]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Embarked'],
      dtype='object')

In [12]:
df.drop(columns=[ "PassengerId", "Name", "Ticket" ], inplace=True)

In [13]:
print("Remaining Columns: ", df.columns)

Remaining Columns:  Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='object')


### Fourth, Convert the categorical columns to numerical
There are variety of techniques available for this like OHE, LabelEncoding, BinaryEncoding

Here, we're using OHE using `pd.get_dummies()` function, Alternate method includes `OneHotEncoder`, `LabelEncoder` of `sklearn` library.

In [22]:
df = pd.get_dummies(df)
for column in df.columns:
    if column.startswith('Sex') or column.startswith('Embarked'):
        df[column] = df[column].astype(np.int8)

In [24]:
print("So Far dataset looks like")
df.sample(10)

So Far dataset looks like


,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
291,1,1,19.000000,1,0,91.0792,1,0,1,0,0
180,0,3,29.699118,8,2,69.5500,1,0,0,0,1
776,0,3,29.699118,0,0,7.7500,0,1,0,1,0
830,1,3,15.000000,1,0,14.4542,1,0,1,0,0
803,1,3,0.420000,0,1,8.5167,0,1,1,0,0
566,0,3,19.000000,0,0,7.8958,0,1,0,0,1
430,1,1,28.000000,0,0,26.5500,0,1,0,0,1
807,0,3,18.000000,0,0,7.7750,1,0,0,0,1
642,0,3,2.000000,3,2,27.9000,1,0,0,0,1
13,0,3,39.000000,1,5,31.2750,0,1,0,0,1


### Five, Creating a custom column name `FamilySize` using `SibSp` and `Parch` column
+ `SibSp` tells no. of siblings or spouses traveling with the passenger.
+ `Parch` tells no. of parents or children traveling with the passenger

`FamilySize` = `SibSp` + `Parch` + 1
<br/>
Why `+1` extra, for passenger who are travelling alone.

In [25]:
df["FamilySize"] = df.SibSp + df.Parch + 1

Once this is done, new we can drop `SibSp` and `Parch` columns as well.

In [26]:
df.drop(columns=['SibSp', 'Parch'], inplace=True)

In [27]:
print("Final dataset after all Preprocessing: ")
df.sample(10)

Final dataset after all Preprocessing: 


,Survived,Pclass,Age,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,FamilySize
87,0,3,29.699118,8.0500,0,1,0,0,1,1
131,0,3,20.000000,7.0500,0,1,0,0,1,1
212,0,3,22.000000,7.2500,0,1,0,0,1,1
841,0,2,16.000000,10.5000,0,1,0,0,1,1
847,0,3,35.000000,7.8958,0,1,1,0,0,1
345,1,2,24.000000,13.0000,1,0,0,0,1,1
204,1,3,18.000000,8.0500,0,1,0,0,1,1
209,1,1,40.000000,31.0000,0,1,1,0,0,1
728,0,2,25.000000,26.0000,0,1,0,0,1,2
601,0,3,29.699118,7.8958,0,1,0,0,1,1
